In [0]:
import pandas as pd
import requests
from io import BytesIO
from pyspark.sql.functions import current_timestamp

# Paste a NEW full SAS token for:
# Storage account: dlliveuberprojectdev
# Container: raw
SAS_TOKEN = "sp=r&st=2026-09-08T10:02:59Z&se=2026-09-08T18:17:59Z&spr=https&sv=2026-02-06&sr=c&sig=cqNxlOJWfX2G0PhMbAtShWQhj0Q6C31%2BObF5P4jDccs%3D"

BASE_URL = "https://dlliveuberprojectdev.blob.core.windows.net/raw"
SOURCE_FOLDER = "ingestion"

files = [
    "map_cities",
    "map_cancellation_reasons",
    "map_payment_methods",
    "map_ride_statuses",
    "map_vehicle_makes",
    "map_vehicle_types",
]

sas = SAS_TOKEN.strip().lstrip("?")

for file_name in files:
    file_url = f"{BASE_URL}/{SOURCE_FOLDER}/{file_name}.json?{sas}"

    print(f"Reading: {file_name}.json")

    response = requests.get(file_url, timeout=60)
    response.raise_for_status()

    df = pd.read_json(BytesIO(response.content))
    print(f"Pandas rows: {len(df)}")

    df_spark = spark.createDataFrame(df)

    # Add the timestamp needed for City SCD processing
    if file_name == "map_cities":
        df_spark = df_spark.withColumn(
            "updated_at",
            current_timestamp()
        )

    print(f"Spark rows: {df_spark.count()}")

    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"uber.bronze.{file_name}")
    )

    print(f"Successfully loaded: uber.bronze.{file_name}")
    print("-" * 60)

Reading: map_cities.json
Pandas rows: 10
Spark rows: 10
Successfully loaded: uber.bronze.map_cities
------------------------------------------------------------
Reading: map_cancellation_reasons.json
Pandas rows: 4
Spark rows: 4
Successfully loaded: uber.bronze.map_cancellation_reasons
------------------------------------------------------------
Reading: map_payment_methods.json
Pandas rows: 4
Spark rows: 4
Successfully loaded: uber.bronze.map_payment_methods
------------------------------------------------------------
Reading: map_ride_statuses.json
Pandas rows: 2
Spark rows: 2
Successfully loaded: uber.bronze.map_ride_statuses
------------------------------------------------------------
Reading: map_vehicle_makes.json
Pandas rows: 7
Spark rows: 7
Successfully loaded: uber.bronze.map_vehicle_makes
------------------------------------------------------------
Reading: map_vehicle_types.json
Pandas rows: 5
Spark rows: 5
Successfully loaded: uber.bronze.map_vehicle_types
-----------------

In [0]:
%sql
select * from uber.bronze.rides_raw

key,value,topic,partition,offset,timestamp,timestampType,rides
null,eyJyaWRlX2lkIjogIjk5ZmM2ZGUxLTJjZTktNDQ4Ni04MGU4LTMyOTcxYjBmYTQ1YyIsICJjb25maXJtYXRpb25fbnVtYmVyIjogIkNHNS05MDgyLUZTNTMiLCAicGFzc2VuZ2VyX2lkIjogIjU4MDNmYTA2LTI4NzMtNGQ2NS05MjM1LWE0ZWQ5OWU0OWM2YiIsICJkcml2ZXJfaWQiOiAiNzAyOGJkYTgtOThhNi00YTM2LTkxNzMtODIzNmMwNDNlZGEyIiwgInZlaGljbGVfaWQiOiAiNTVkZmY0ZDgtNGE1MS00NDZlLTg5YzEtNGMyODEwOTJiM2I0IiwgInBpY2t1cF9sb2NhdGlvbl9pZCI6ICJmYjY3OTQxYi03ZjdmLTRhNDYtYTZjZS0yMzE5YzZjMzBlNDgiLCAiZHJvcG9mZl9sb2NhdGlvbl9pZCI6ICIxYWI3OWZhNy1hMDliLTQ1MDItOWUxMS04YmI4NzRjMWQwZTYiLCAidmVoaWNsZV90eXBlX2lkIjogNSwgInZlaGljbGVfbWFrZV9pZCI6IDcsICJwYXltZW50X21ldGhvZF9pZCI6IDMsICJyaWRlX3N0YXR1c19pZCI6IDEsICJwaWNrdXBfY2l0eV9pZCI6IDYsICJkcm9wb2ZmX2NpdHlfaWQiOiA3LCAiY2FuY2VsbGF0aW9uX3JlYXNvbl9pZCI6IDQsICJwYXNzZW5nZXJfbmFtZSI6ICJSYWNoZWwgWW91bmciLCAicGFzc2VuZ2VyX2VtYWlsIjogIndpbGxpYW1zbWVnYW5AZXhhbXBsZS5uZXQiLCAicGFzc2VuZ2VyX3Bob25lIjogIjU2MS40ODkuMTMxOXgyMjAiLCAiZHJpdmVyX25hbWUiOiAiQ3JhaWcgRHVuY2FuIiwgImRyaXZlcl9yYXRpbmciOiA0Ljk1LCAiZHJpdmVyX3Bob25lIjogIisxLTU5Ni03MjctMzY2NHg2NDg1OSIsICJkcml2ZXJfbGljZW5zZSI6ICJ6aS1Xa3ctNTY4ODYwOCIsICJ2ZWhpY2xlX21vZGVsIjogIldlZWsiLCAidmVoaWNsZV9jb2xvciI6ICJXaGl0ZSIsICJsaWNlbnNlX3BsYXRlIjogIlB4Si04ODI5IiwgInBpY2t1cF9hZGRyZXNzIjogIjgwMyBMaXNhIFZpYSBBcHQuIDYwOCwgTGVldG93biwgS1MgNTc4NzgiLCAicGlja3VwX2xhdGl0dWRlIjogMzUuNDc1ODc5LCAicGlja3VwX2xvbmdpdHVkZSI6IDM5LjYxNjUxNiwgImRyb3BvZmZfYWRkcmVzcyI6ICI4NzU3IEtyaXN0eSBGb3JrIFN1aXRlIDgwNywgUG9ydCBQYXVsLCBOTSA3NzQzNCIsICJkcm9wb2ZmX2xhdGl0dWRlIjogNTEuOTMyMzI3LCAiZHJvcG9mZl9sb25naXR1ZGUiOiAxMC42NTQyMTQsICJkaXN0YW5jZV9taWxlcyI6IDYuMDgsICJkdXJhdGlvbl9taW51dGVzIjogMTAyLCAiYm9va2luZ190aW1lc3RhbXAiOiAiMjAyNi0wOC0yM1QyMzoyMjowNS4wNDAxNDIiLCAicGlja3VwX3RpbWVzdGFtcCI6ICIyMDI2LTA4LTIzVDIzOjIzOjA1LjA0MDE0MiIsICJkcm9wb2ZmX3RpbWVzdGFtcCI6ICIyMDI2LTA4LTI0VDAxOjA1OjA1LjA0MDE0MiIsICJiYXNlX2ZhcmUiOiAyLjUsICJkaXN0YW5jZV9mYXJlIjogMTAuNjQsICJ0aW1lX2ZhcmUiOiAzNS43LCAic3VyZ2VfbXVsdGlwbGllciI6IDEuMjgsICJzdWJ0b3RhbCI6IDYyLjUyLCAidGlwX2Ftb3VudCI6IDMsICJ0b3RhbF9mYXJlIjogNjUuNTIsICJyYXRpbmciOiBudWxsfQ==,ubertopic,0,0,2026-09-02T06:53:08.323Z,0,"{""ride_id"": ""99fc6de1-2ce9-4486-80e8-32971b0fa45c"", ""confirmation_number"": ""CG5-9082-FS53"", ""passenger_id"": ""5803fa06-2873-4d65-9235-a4ed99e49c6b"", ""driver_id"": ""7028bda8-98a6-4a36-9173-8236c043eda2"", ""vehicle_id"": ""55dff4d8-4a51-446e-89c1-4c281092b3b4"", ""pickup_location_id"": ""fb67941b-7f7f-4a46-a6ce-2319c6c30e48"", ""dropoff_location_id"": ""1ab79fa7-a09b-4502-9e11-8bb874c1d0e6"", ""vehicle_type_id"": 5, ""vehicle_make_id"": 7, ""payment_method_id"": 3, ""ride_status_id"": 1, ""pickup_city_id"": 6, ""dropoff_city_id"": 7, ""cancellation_reason_id"": 4, ""passenger_name"": ""Rachel Young"", ""passenger_email"": ""williamsmegan@example.net"", ""passenger_phone"": ""561.489.1319x220"", ""driver_name"": ""Craig Duncan"", ""driver_rating"": 4.95, ""driver_phone"": ""+1-596-727-3664x64859"", ""driver_license"": ""zi-Wkw-5688608"", ""vehicle_model"": ""Week"", ""vehicle_color"": ""White"", ""license_plate"": ""PxJ-8829"", ""pickup_address"": ""803 Lisa Via Apt. 608, Leetown, KS 57878"", ""pickup_latitude"": 35.475879, ""pickup_longitude"": 39.616516, ""dropoff_address"": ""8757 Kristy Fork Suite 807, Port Paul, NM 77434"", ""dropoff_latitude"": 51.932327, ""dropoff_longitude"": 10.654214, ""distance_miles"": 6.08, ""duration_minutes"": 102, ""booking_timestamp"": ""2026-08-23T23:22:05.040142"", ""pickup_timestamp"": ""2026-08-23T23:23:05.040142"", ""dropoff_timestamp"": ""2026-08-24T01:05:05.040142"", ""base_fare"": 2.5, ""distance_fare"": 10.64, ""time_fare"": 35.7, ""surge_multiplier"": 1.28, ""subtotal"": 62.52, ""tip_amount"": 3, ""total_fare"": 65.52, ""rating"": null}"
null,eyJyaWRlX2lkIjogIjAzYzg4MzY5LWEwMTQtNDRhMS1iZGRjLTcxZTJhMjU0YzAxYSIsICJjb25maXJtYXRpb25fbnVtYmVyIjogIkpiMi02OTY2LWNqMzMiLCAicGFzc2VuZ2VyX2lkIjogIjZlNTc5OGRhLWJmZWYtNDU3MC1hYTYzLWQ0NjE4OTg2OTBjMCIsICJkcml2ZXJfaWQiOiAiYmY1YTA0YzctY2M2OC00OTVjLWFmMGEtOTU5ZGRkMDJjZDdlIiwgInZlaGljb

In [0]:
display(spark.sql("SHOW TABLES IN uber.bronze"))

database,tableName,isTemporary
bronze,map_cancellation_reasons,false
bronze,map_cities,false
bronze,map_payment_methods,false
bronze,map_ride_statuses,false
bronze,map_vehicle_makes,false
bronze,map_vehicle_types,false
bronze,rides_raw,false


In [0]:
%sql
DESCRIBE TABLE uber.bronze.map_cities;

col_name,data_type,comment
city_id,bigint,null
city,string,null
state,string,null
region,string,null
updated_at,timestamp,null


In [0]:
%sql
select * from uber.bronze.map_cities

city_id,city,state,region,updated_at
1,New York,NY,Northeast,2026-09-08T10:19:31.072Z
2,Los Angelas,CA,West,2026-09-08T10:19:31.072Z
3,Chicago,IL,Midwest,2026-09-08T10:19:31.072Z
4,Houston,TX,South,2026-09-08T10:19:31.072Z
5,Phoenix,AZ,Southwest,2026-09-08T10:19:31.072Z
6,Philadelphia,PA,Northeast,2026-09-08T10:19:31.072Z
7,San Antonio,TX,South,2026-09-08T10:19:31.072Z
8,San Diego,CA,West,2026-09-08T10:19:31.072Z
9,Dallas,TX,South,2026-09-08T10:19:31.072Z
10,San Jose,CA,West,2026-09-08T10:19:31.072Z


In [0]:
%sql
select * from uber.bronze.silver_obt_streaming;


ride_id,confirmation_number,passenger_id,driver_id,vehicle_id,pickup_location_id,dropoff_location_id,vehicle_type_id,vehicle_make_id,payment_method_id,ride_status_id,pickup_city_id,dropoff_city_id,cancellation_reason_id,passenger_name,passenger_email,passenger_phone,driver_name,driver_rating,driver_phone,driver_license,vehicle_model,vehicle_color,license_plate,pickup_address,pickup_latitude,pickup_longitude,dropoff_address,dropoff_latitude,dropoff_longitude,distance_miles,duration_minutes,booking_timestamp,pickup_timestamp,dropoff_timestamp,base_fare,distance_fare,time_fare,surge_multiplier,subtotal,tip_amount,total_fare,rating,vehicle_make,vehicle_type,vehicle_type_description,base_rate,per_mile,per_minute,ride_status,payment_method,is_card,requires_auth,pickup_city,pickup_state,pickup_region,city_updated_at,cancellation_reason
35fe5cb8-5ab1-4e65-9582-439fecaee1f1,Zs6-8714-GD09,f8861a0b-186d-4700-9602-744b8465b064,adc232b3-b0bf-4a69-92eb-7cb5c30f750a,c89ee9d5-3862-4f2c-a22f-9189a7bf7195,21e4cd72-7d87-424b-a793-8395b0458f47,f3049ab1-5df6-4d1d-be95-86d7d9aaab13,1,6,4,2,6,3,4,Mr. Raymond Compton,danny47@example.net,8846567600,Ariana Lester,4.6,+1-874-479-3270x06125,Wk-KjK-7699834,Section,Red,eAS-0731,"6499 Mark Stream, West Robert, MH 96356",-83.816729,157.810961,"11599 Julian Rest Suite 050, Smithchester, ID 45716",-0.060372,79.994693,28.08,75,2026-08-26T14:14:20.955Z,2026-08-26T14:22:20.955856,2026-08-26T15:37:20.955856,2.5,49.14,26.25,1.69,131.63,16.34,147.97,5.0,BMW,UberX,Standard,2.5,1.75,0.35000000000000003,Cancelled,Cash,false,false,Philadelphia,PA,Northeast,2026-09-08T10:19:31.072Z,null
ab203c5e-c0f9-455f-88b6-db7404dd78d0,zc4-4367-AE43,ec5aaf39-10a5-411f-b40f-d48fc814ae99,633353cd-7cba-406f-8c72-788b24a998cd,f7be073c-9bdd-4f5f-8c34-5424fb181625,0b160d07-506c-4897-bb1f-f6f8356c8cd3,02fd14f0-f58b-459f-be91-780fc2ce1adc,3,7,1,2,4,10,4,Matthew Romero,jon17@example.org,353-873-9456,Alexis Pugh,4.83,525.774.3493x755,iH-jjJ-8218163,Particular,Red,BPd-2694,"2019 Daniel Mount Apt. 611, Lake Andrewburgh, NM 40608",62.623548,48.844977,"838 Garza Wells Apt. 430, Johnberg, ME 62196",-81.717322,79.067511,1.41,114,2026-08-17T12:21:38.837Z,2026-08-17T12:26:38.837331,2026-08-17T14:20:38.837331,2.5,2.47,39.9,2.37,106.34,0.0,106.34,1.0,Mercedes,UberPOOL,Shared Ride,2.0,1.5,0.30000000000000004,Cancelled,Credit Card,true,true,Houston,TX,South,2026-09-08T10:19:31.072Z,null


In [0]:
%sql
select passenger_name,
        passenger_email,
        passenger_phone,
        passenger_id
 from uber.bronze.silver_obt    


passenger_name,passenger_email,passenger_phone,passenger_id
Sarah Morgan,gabriela88@example.com,292.404.5825x11055,c7130b66-d4dc-4b47-a9a7-446e45afc81d
Rachel Young,williamsmegan@example.net,561.489.1319x220,5803fa06-2873-4d65-9235-a4ed99e49c6b
Victor Edwards,prestonandersen@example.com,579.785.6524,6e5798da-bfef-4570-aa63-d461898690c0
Joshua Ray,kturner@example.net,847-503-3846x5170,05c2b5ff-f8fd-4018-8b8d-5e9002003e11
Shannon Nelson,emeza@example.org,+1-601-909-2749x3316,a1e01eae-e966-40eb-811d-66217e1fcf20
Matthew Romero,jon17@example.org,353-873-9456,ec5aaf39-10a5-411f-b40f-d48fc814ae99
